<div dir="rtl">

# مشروع صغير: تحليل طلبات مطعم بمساعدة الذكاء الاصطناعي
**السؤال الرئيسي:** أين تربح إدارة المطعم، ومتى يزدحم، وما الذي يُغضب العملاء؟

**البيانات:** `restaurant_orders.csv` (طلبات 3 فروع، يناير إلى يونيو 2025). اقرأ `data_dictionary_restaurant.md` أولًا.

**المهارات المستخدمة في Cursor:** `/data-quality-report` ← `/kpi-summary` ← `/clear-charts` ← `/insight-report`

**طريقة العمل:** لكل سؤال: اطلب من Agent الكود (أو استدعِ المهارة)، اقرأ الكود قبل قبوله، شغّله، ثم قارن بنقطة التحقق.

</div>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("restaurant_orders.csv")
print(df.shape)   # نقطة تحقق: (1515, 12)
df.head()

(1515, 12)


,order_id,order_date,order_hour,branch,channel,category,item,quantity,unit_price,total,delivery_minutes,rating
0,R10001,2025-01-01,11,Downtown,Delivery,Sides,Hummus,5,3.5,17.5,43.0,3.0
1,R10002,2025-01-01,19,Mall,Takeaway,Main,Falafel Wrap,4,4.5,18.0,NaN,4.0
2,R10003,2025-01-01,20,Downtown,Dine-in,Main,Chicken Shawarma,3,6.5,19.5,NaN,NaN
3,R10004,2025-01-01,22,University,Delivery,Main,Beef Burger,2,8.0,16.0,28.0,5.0
4,R10005,2025-01-02,11,Mall,Delivery,Drinks,Fresh Juice,3,3.0,9.0,29.0,4.0


<div dir="rtl">

## الخطوة 1: فحص الجودة والتنظيف
في Agent: `/data-quality-report افحص restaurant_orders.csv` ثم: «نظّف البيانات حسب التقرير واحفظها في clean».

**نقطة تحقق:** 15 صفًا مكررًا، و6 كتابات مختلفة لأسماء الفروع، و136 تقييمًا مفقودًا. بعد التنظيف: **1500 صف** و3 فروع فقط.

</div>

In [2]:
# فحص جودة البيانات: لا نعدل الملف الأصلي، فقط نتحقق ونقترح العلاج

df = pd.read_csv("restaurant_orders.csv")

print("1) عدد الصفوف والأعمدة:")
print(df.shape)

print("\n2) أنواع الأعمدة:")
print(df.dtypes.to_string())

print("\n3) القيم المفقودة في كل عمود:")
print(df.isna().sum().to_string())

print("\n4) عدد الصفوف المكررة بالكامل:")
print(df.duplicated().sum())

print("\n5) القيم المختلفة للأعمدة النصية ذات القيم القليلة:")
for col in df.select_dtypes(include=["object"]).columns:
    n_unique = df[col].nunique(dropna=False)
    if n_unique < 20:
        print(f"\nالعمود: {col} | عدد القيم المختلفة: {n_unique}")
        print(pd.Series(df[col].dropna().unique()).to_string(index=False))

print("\n6) القيم الرقمية: الحد الأدنى والحد الأقصى + تنبيه القيم السالبة:")
for col in df.select_dtypes(include=["number"]).columns:
    s = df[col]
    print(f"\nالعمود: {col} | الحد الأدنى: {s.min()} | الحد الأقصى: {s.max()}")
    neg_count = (s < 0).sum()
    if neg_count > 0:
        print(f"تنبيه: توجد {neg_count} قيمة(ات) سالبة غير منطقية في {col}.")

# تقرير جودة البيانات بالعربية
branch_rows = df["branch"].astype(str).str.strip()
branch_mismatch = (~branch_rows.str.title().isin(["Downtown", "Mall", "University"])).sum()

quality_report = pd.DataFrame(
    [
        ["صفوف مكررة بالكامل", "all columns", int(df.duplicated().sum()), "حذف التكرارات بعد التأكد من عدم وجود بيانات مكررة من مصدر موثوق، مع الاحتفاظ بسجل التكرار."],
        ["كتابة غير موحدة في اسم الفرع", "branch", int(branch_mismatch), "توحيد أسماء الفروع إلى Downtown / Mall / University باستخدام strip() وtitle() أو خريطة تحويل واضحة."],
        ["تقييمات مفقودة", "rating", int(df["rating"].isna().sum()), "عدم حذف الصفوف، فقط استبعاد القيم المفقودة عند حساب متوسط التقييم وذكر ذلك في التحليل."],
    ],
    columns=["المشكلة", "العمود", "عدد الصفوف", "اقتراح المعالجة"]
)

print("\n7) تقرير جودة البيانات المقترح:")
print(quality_report.to_string(index=False))

1) عدد الصفوف والأعمدة:
(1515, 12)

2) أنواع الأعمدة:
order_id             object
order_date           object
order_hour            int64
branch               object
channel              object
category             object
item                 object
quantity              int64
unit_price          float64
total               float64
delivery_minutes    float64
rating              float64

3) القيم المفقودة في كل عمود:
order_id               0
order_date             0
order_hour             0
branch                 0
channel                0
category               0
item                   0
quantity               0
unit_price             0
total                  0
delivery_minutes    1137
rating               136

4) عدد الصفوف المكررة بالكامل:
15

5) القيم المختلفة للأعمدة النصية ذات القيم القليلة:

العمود: branch | عدد القيم المختلفة: 9
   Downtown
       Mall
 University
       mall
 university
       MALL
   DOWNTOWN
   downtown
 UNIVERSITY

العمود: channel | عدد القيم المختلفة: 3
De

In [3]:
# تنظيف البيانات: إزالة الصفوف المكررة وتوحيد أسماء الفروع
# لا نعدل الملف الأصلي، بل نُنشئ نسخة نظيفة باسم clean

df = pd.read_csv("restaurant_orders.csv")
clean = df.drop_duplicates().copy()

# توحيد أسماء الفروع: إزالة المسافات وتحويلها إلى أسماء موحدة
branch_map = {
    "downtown": "Downtown",
    "mall": "Mall",
    "university": "University",
}

clean["branch"] = (
    clean["branch"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(branch_map)
    .fillna(clean["branch"].astype(str).str.strip().str.title())
)

# التحقق من النتيجة النهائية
print("بعد التنظيف:", clean.shape)
print("أعداد الفروع:")
print(clean["branch"].value_counts().to_string())
print("\nالقيم المفقودة بعد التنظيف:")
print(clean.isna().sum().to_string())

# احتفظ بالنسخة النظيفة في متغير باسم clean لاستخدامها في التحليل لاحقًا
clean.head()

بعد التنظيف: (1500, 12)
أعداد الفروع:
branch
Mall          590
Downtown      466
University    444

القيم المفقودة بعد التنظيف:
order_id               0
order_date             0
order_hour             0
branch                 0
channel                0
category               0
item                   0
quantity               0
unit_price             0
total                  0
delivery_minutes    1126
rating               132


,order_id,order_date,order_hour,branch,channel,category,item,quantity,unit_price,total,delivery_minutes,rating
0,R10001,2025-01-01,11,Downtown,Delivery,Sides,Hummus,5,3.5,17.5,43.0,3.0
1,R10002,2025-01-01,19,Mall,Takeaway,Main,Falafel Wrap,4,4.5,18.0,NaN,4.0
2,R10003,2025-01-01,20,Downtown,Dine-in,Main,Chicken Shawarma,3,6.5,19.5,NaN,NaN
3,R10004,2025-01-01,22,University,Delivery,Main,Beef Burger,2,8.0,16.0,28.0,5.0
4,R10005,2025-01-02,11,Mall,Delivery,Drinks,Fresh Juice,3,3.0,9.0,29.0,4.0


<div dir="rtl">

## الخطوة 2: مؤشرات الأداء (س1)
**س1:** ما إجمالي الإيرادات وعدد الطلبات ومتوسط قيمة الطلب ومتوسط التقييم؟ وما هي لكل فرع؟

في Agent: `/kpi-summary احسب المؤشرات العامة ثم قارن الفروع`

</div>

In [4]:
# مؤشرات الأداء العامة بعد التنظيف
# نستخدم البيانات في clean التي تم إنشاؤها في الخلية السابقة

overall_total = clean["total"].sum()
overall_orders = len(clean)
avg_order_value = overall_total / overall_orders if overall_orders else 0
avg_rating = clean["rating"].dropna().mean() if clean["rating"].notna().any() else 0

overall_kpis = pd.DataFrame(
    {
        "المؤشر": ["إجمالي الإيرادات", "عدد الطلبات", "متوسط قيمة الطلب", "متوسط التقييم"],
        "القيمة": [overall_total, overall_orders, avg_order_value, avg_rating],
    }
)

overall_kpis["القيمة"] = overall_kpis["القيمة"].round(2)
print("المؤشرات العامة:")
print(overall_kpis.to_string(index=False))

# مقارنة المؤشرات لكل فرع
branch_kpis = (
    clean.groupby("branch", as_index=False)
    .agg(
        إجمالي_الإيرادات=("total", "sum"),
        عدد_الطلبات=("order_id", "count"),
        متوسط_قيمة_الطلب=("total", "mean"),
        متوسط_التقييم=("rating", "mean"),
    )
    .sort_values("إجمالي_الإيرادات", ascending=False)
    .reset_index(drop=True)
)

branch_kpis["النسبة_من_الإجمالي_%"] = (
    branch_kpis["إجمالي_الإيرادات"] / overall_total * 100
).round(2)

for col in ["إجمالي_الإيرادات", "متوسط_قيمة_الطلب", "متوسط_التقييم"]:
    branch_kpis[col] = branch_kpis[col].round(2)

print("\nمقارنة الفروع:")
print(branch_kpis.to_string(index=False))

# سطر تحقق: مجموع الإيرادات في جدول المقارنة يساوي الإجمالي العام
check_total = branch_kpis["إجمالي_الإيرادات"].sum()
assert abs(check_total - overall_total) < 1e-9, "مجموع الإيرادات في جدول المقارنة لا يساوي الإجمالي العام"
print("\nسطر التحقق: مجموع الإيرادات في جدول المقارنة يساوي الإجمالي العام.")

# ثلاثة أسطر عربية فقط
best_branch = branch_kpis.iloc[0]
lowest_branch = branch_kpis.iloc[-1]
diff_value = best_branch["إجمالي_الإيرادات"] - lowest_branch["إجمالي_الإيرادات"]

print(f"أعلى فرع: {best_branch['branch']} بإيرادات {best_branch['إجمالي_الإيرادات']:.2f} دولار.")
print(f"أقل فرع: {lowest_branch['branch']} بإيرادات {lowest_branch['إجمالي_الإيرادات']:.2f} دولار.")
print(f"الفارق بينهما: {diff_value:.2f} دولار.")

المؤشرات العامة:
          المؤشر   القيمة
إجمالي الإيرادات 18751.00
     عدد الطلبات  1500.00
متوسط قيمة الطلب    12.50
   متوسط التقييم     4.07

مقارنة الفروع:
    branch  إجمالي_الإيرادات  عدد_الطلبات  متوسط_قيمة_الطلب  متوسط_التقييم  النسبة_من_الإجمالي_%
      Mall            7933.0          590             13.45           4.12                 42.31
  Downtown            6678.0          466             14.33           4.01                 35.61
University            4140.0          444              9.32           4.04                 22.08

سطر التحقق: مجموع الإيرادات في جدول المقارنة يساوي الإجمالي العام.
أعلى فرع: Mall بإيرادات 7933.00 دولار.
أقل فرع: University بإيرادات 4140.00 دولار.
الفارق بينهما: 3793.00 دولار.


<div dir="rtl">

## الخطوة 3: الذروة والقنوات والأصناف (س2، س3، س4)
- **س2:** ما ساعات الذروة؟ وأي أيام الأسبوع أكثر ازدحامًا؟
- **س3:** ما حصة كل قناة (صالة، سفري، توصيل) من الإيرادات؟
- **س4:** ما أعلى 5 أصناف من حيث الإيرادات؟

ارسم رسمين على الأقل باستخدام `/clear-charts`.

</div>

In [ ]:
# س2 هنا


In [ ]:
# س3 وس4 هنا


<div dir="rtl">

## الخطوة 4: رضا العملاء (س5)
**س5:** هل يرتبط وقت التوصيل بتقييم العميل؟
قسّم طلبات التوصيل إلى ثلاث فئات: أقل من 30 دقيقة، من 30 إلى 45، أكثر من 45. ثم احسب عدد الطلبات ومتوسط التقييم لكل فئة.

> هذه مقارنة بين متوسطات، لا تحتاج أي نموذج تعلم آلي. لكنها لا تثبت السبب وحدها.

</div>

In [ ]:
# س5 هنا


<div dir="rtl">

## الخطوة 5: الرؤى والتوصيات (س6)
في Agent: `/insight-report اكتب الرؤى والتوصيات من نتائج هذا الدفتر واحفظها في insights.md`

ثم راجع: هل كل رقم في التقرير موجود فعلًا في مخرجات الدفتر؟ سجّل أي خطأ في سجل التحقق أدناه.

## سجل التحقق
| ماذا قال الذكاء الاصطناعي | كيف تحققت | النتيجة |
| --- | --- | --- |
|  |  |  |

</div>